# Batch Digester Analysis (D1–D4)

Reads the data saved by `pull_data.py` (no PI access here), splits it into batches, calculates the H-factor per batch and compares the 4 digesters.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config
from pull_data import load_data
from batch import all_phase_tables, build_batch_table
from hfactor import h_factor_per_batch

sns.set_theme(style="whitegrid")

## 1. Load saved data

In [2]:
DATA_NAME = f"digester_{config.START_TIME[:10]}_{config.END_TIME[:10]}"
df = load_data(DATA_NAME)
print(df.shape, df.index.min(), df.index.max())
df.head()

FileNotFoundError: [Errno 2] No such file or directory: 'data\\digester_2026-06-01_2026-10-01.csv.gz'

## 2. Phase table and batch table

In [ ]:
phases = all_phase_tables(df)
batches = build_batch_table(phases)

print("Phases seen:", phases["phase"].unique())
print("Batches per digester:")
print(batches["digester"].value_counts().sort_index())
batches.head()

## 3. H-factor per batch

In [ ]:
hf = h_factor_per_batch(df, phases, config.H_FACTOR_PHASES)
batches = batches.merge(hf, on=["digester", "batch_id"], how="left")
batches[["digester", "batch_id", "batch_no", "start", "h_factor", "max_temp"]].head()

## 4. Volatility per digester (Mean, SD, CV%)

In [ ]:
def volatility_table(batches, col):
    g = batches.groupby("digester")[col]
    table = pd.DataFrame({"N": g.count(), "Mean": g.mean(), "SD": g.std(),
                          "Min": g.min(), "Max": g.max()})
    table["CV%"] = 100 * table["SD"] / table["Mean"]
    return table.round(2)

for col in ["h_factor", "max_temp", "min_HEAT&COOK", "min_PREHYDRAU"]:
    if col in batches.columns:
        print(f"\n{col}")
        display(volatility_table(batches, col))

## 5. H-factor by digester (boxplot)

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=batches, x="digester", y="h_factor")
plt.title("H-factor per batch by digester")
plt.xlabel("Digester")
plt.ylabel("H-factor")
plt.show()

## 6. H-factor trend over time

In [ ]:
plt.figure(figsize=(12, 5))
for dig in [1, 2, 3, 4]:
    b = batches[batches["digester"] == dig]
    plt.plot(b["start"], b["h_factor"], marker=".", linestyle="-", label=f"D{dig}")
plt.legend()
plt.title("H-factor per batch over time")
plt.ylabel("H-factor")
plt.show()